In [ ]:
# CELL 0 - Install the extra packages once (Colab / laptop). Restart the runtime if pip asks you to.
!pip install -q datasets scikit-image opencv-python-headless

In [ ]:
# CELL 1 - Setup: imports, one Config object holding every setting, and reproducibility. Change numbers here only.
import os, math, time, json
from dataclasses import dataclass
from pathlib import Path
import cv2, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import deform_conv2d

@dataclass
class Config:
    work_dir: str = "/content/work" if os.path.exists("/content") else "./work"  # data + checkpoints live here
    scale: int = 4              # super-resolution factor (28x28 -> 112x112)
    hr_size: int = 112          # HR crop size = ArcFace input size
    n_val: int = 1000           # fixed validation images (pick the best epoch with these)
    n_test: int = 2000          # fixed test images (used once, at the end)
    feat: int = 48              # feature channels of the network
    blocks: int = 6             # number of residual blocks
    keep_ratio: float = 0.5     # fraction of pixels kept as attention keys (the "sparse" part)
    batch: int = 64
    epochs: int = 30
    lr: float = 2e-4
    num_workers: int = 4        # use 0 on Windows notebooks
    seed: int = 42

cfg = Config()
Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)      # reproducible weights and splits
torch.backends.cudnn.benchmark = True                       # faster convolutions for fixed input sizes
print("device:", device, "| work_dir:", cfg.work_dir)

In [ ]:
# CELL 2 - Data: download FFHQ 128x128 (70,000 faces, ~2 GB) once and cache it as ONE uint8 array on disk.
class FFHQCache:
    def __init__(self, cfg):
        self.path = Path(cfg.work_dir) / "ffhq128_u8.npy"           # cached array, shape (N,128,128,3)
    def build(self):
        if self.path.exists(): return                                # already cached -> skip the download
        from datasets import load_dataset
        ds = load_dataset("nuwandaa/ffhq128", split="train")         # 70k faces from Hugging Face
        arr = np.lib.format.open_memmap(self.path, mode="w+", dtype=np.uint8, shape=(len(ds), 128, 128, 3))
        for i, row in enumerate(ds):
            img = row["image"].convert("RGB")
            arr[i] = np.asarray(img if img.size == (128, 128) else img.resize((128, 128), Image.BICUBIC))
        arr.flush()
    def load(self):
        return np.load(self.path, mmap_mode="r")                     # memory-mapped: does not load 3.4 GB into RAM

cache = FFHQCache(cfg); cache.build(); hr_all = cache.load()
print("faces cached:", hr_all.shape)

In [ ]:
# CELL 3 - Degradation: turns a clean HR face into a CCTV-like LR face (blur -> downsample -> noise -> JPEG).
class Degrader:
    def __init__(self, scale=4, blur_p=0.7, noise=(2, 15), jpeg=(30, 70)):
        self.scale, self.blur_p, self.noise, self.jpeg = scale, blur_p, noise, jpeg
    def __call__(self, hr, rng=None):                                # hr: uint8 HxWx3  ->  lr: uint8 (H/s)x(W/s)x3
        rng = rng or np.random.default_rng()                         # seeded rng => reproducible val/test images
        img = hr
        if rng.random() < self.blur_p:                               # 1) random Gaussian blur (lens / motion blur)
            img = cv2.GaussianBlur(img, (int(rng.choice([3, 5, 7])),) * 2, float(rng.uniform(0.2, 2.0)))
        h, w = hr.shape[:2]
        lr = np.asarray(Image.fromarray(img).resize((w // self.scale, h // self.scale), Image.BICUBIC))  # 2) bicubic downsample (antialiased)
        lr = lr + rng.normal(0, rng.uniform(*self.noise), lr.shape)  # 3) sensor noise
        lr = np.clip(lr, 0, 255).astype(np.uint8)
        ok, buf = cv2.imencode(".jpg", lr, [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(*self.jpeg))])    # 4) JPEG compression artifacts
        return cv2.imdecode(buf, cv2.IMREAD_COLOR)

In [ ]:
# CELL 4 - Datasets: fixed train/val/test split. Train images are degraded on the fly; val/test are degraded ONCE with fixed seeds.
def to_tensor(a):                                                    # HWC uint8 -> CHW float in [0,1]
    return torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1).float().div(255)

class FaceTrainSet(Dataset):
    def __init__(self, path, ids, degrader, hr_size):
        self.path, self.ids, self.deg, self.s, self.hr = path, ids, degrader, hr_size, None
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        if self.hr is None: self.hr = np.load(self.path, mmap_mode="r")   # opened lazily inside each worker process
        img, rng = np.asarray(self.hr[self.ids[i]]), np.random.default_rng()
        y, x = rng.integers(0, img.shape[0] - self.s + 1, 2)                # random crop jitter (augmentation)
        crop = img[y:y + self.s, x:x + self.s]
        if rng.random() < 0.5: crop = crop[:, ::-1]                          # random horizontal flip
        crop = np.ascontiguousarray(crop)
        return to_tensor(self.deg(crop, rng)), to_tensor(crop)               # (LR input, HR target)

class FixedEvalSet:                                                  # same LR images every time -> fair comparison of models
    def __init__(self, hr, ids, degrader, hr_size, seed):
        o = (hr.shape[1] - hr_size) // 2                             # centre crop
        crops = [np.asarray(hr[i])[o:o + hr_size, o:o + hr_size] for i in ids]
        lrs = [degrader(c, np.random.default_rng(seed + k)) for k, c in enumerate(crops)]   # per-image fixed seed
        self.hr = torch.from_numpy(np.stack(crops)).permute(0, 3, 1, 2).contiguous()        # uint8 tensors (small in RAM)
        self.lr = torch.from_numpy(np.stack(lrs)).permute(0, 3, 1, 2).contiguous()
    def __len__(self): return len(self.hr)
    def batches(self, bs):
        for i in range(0, len(self), bs):
            yield self.lr[i:i + bs].float() / 255, self.hr[i:i + bs].float() / 255

perm = np.random.default_rng(cfg.seed).permutation(len(hr_all))     # one fixed shuffle -> disjoint splits
test_ids, val_ids, train_ids = perm[:cfg.n_test], perm[cfg.n_test:cfg.n_test + cfg.n_val], perm[cfg.n_test + cfg.n_val:]
deg = Degrader(cfg.scale)
val_set, test_set = FixedEvalSet(hr_all, val_ids, deg, cfg.hr_size, 1000), FixedEvalSet(hr_all, test_ids, deg, cfg.hr_size, 2000)
train_loader = DataLoader(FaceTrainSet(cache.path, train_ids, deg, cfg.hr_size), batch_size=cfg.batch, shuffle=True,
                          num_workers=cfg.num_workers, drop_last=True, pin_memory=(device == "cuda"),
                          persistent_workers=cfg.num_workers > 0)
print(f"train {len(train_ids)} | val {len(val_set)} | test {len(test_set)} | LR {tuple(test_set.lr.shape[1:])} -> HR {tuple(test_set.hr.shape[1:])}")
k = 6                                                                # visual check: top = LR (enlarged), bottom = HR
ups = [cv2.resize(test_set.lr[i].permute(1, 2, 0).numpy(), (cfg.hr_size,) * 2, interpolation=cv2.INTER_NEAREST) for i in range(k)]
Image.fromarray(np.concatenate([np.concatenate(ups, 1), np.concatenate([test_set.hr[i].permute(1, 2, 0).numpy() for i in range(k)], 1)], 0))

In [ ]:
# CELL 5 - Model: sparse sampling conv + adaptive sparse linear attention, stacked in residual blocks. Flags allow ablations.
class SparseSamplingConv(nn.Module):
    """9 learnable sample points per pixel (deformable-style), kept inside a 7x7 window."""
    def __init__(self, c, max_offset=2.0):
        super().__init__()
        self.max_offset = max_offset
        self.offset = nn.Conv2d(c, 18, 3, padding=1)                  # predicts (dy, dx) for each of the 9 points
        nn.init.zeros_(self.offset.weight); nn.init.zeros_(self.offset.bias)   # start exactly as a normal 3x3 conv
        self.weight = nn.Parameter(torch.empty(c, c, 3, 3)); nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.bias = nn.Parameter(torch.zeros(c))
    def forward(self, x):
        off = torch.tanh(self.offset(x)) * self.max_offset            # 1 (grid) + 2 (offset) = 3 px -> 7x7 window
        return deform_conv2d(x, off, self.weight, self.bias, padding=(1, 1))

class AdaptiveSparseLinearAttention(nn.Module):
    """Linear attention O(n) where only the top-k most important pixels act as keys (content-adaptive sparsity)."""
    def __init__(self, c, keep_ratio=0.5):
        super().__init__()
        self.keep_ratio = keep_ratio
        self.qkv = nn.Conv2d(c, 3 * c, 1)
        self.score = nn.Sequential(nn.Conv2d(c, c // 4, 1), nn.GELU(), nn.Conv2d(c // 4, 1, 1))  # per-pixel importance
        self.proj = nn.Conv2d(c, c, 1)
    def key_mask(self, x):
        s = torch.sigmoid(self.score(x)).flatten(1)                   # (N, HW) importance in (0,1)
        k = max(1, int(self.keep_ratio * s.shape[1]))
        hard = torch.zeros_like(s).scatter_(1, s.topk(k, dim=1).indices, 1.0)   # 1 for the top-k pixels, 0 elsewhere
        return hard + s - s.detach()                                  # forward = hard mask, backward = gradient of s (straight-through)
    def forward(self, x):
        n, c, h, w = x.shape
        q, k, v = self.qkv(x).flatten(2).chunk(3, dim=1)              # each (N, C, HW)
        q = F.elu(q) + 1                                              # positive feature map phi(q)
        k = (F.elu(k) + 1) * self.key_mask(x).unsqueeze(1)            # phi(k), zeroed for non-selected pixels
        kv = torch.einsum("ncl,ndl->ncd", k, v)                       # (N,C,C) global summary, computed once -> linear cost
        z = 1.0 / (torch.einsum("ncl,nc->nl", q, k.sum(2)) + 1e-6)    # normalisation per query pixel
        out = torch.einsum("ncd,ncl,nl->ndl", kv, q, z).reshape(n, c, h, w)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self, c, use_ssc=True, use_attn=True, keep_ratio=0.5):
        super().__init__()
        self.n1 = nn.GroupNorm(1, c)                                  # LayerNorm-style (no BatchNorm in SR)
        self.spatial = SparseSamplingConv(c) if use_ssc else nn.Conv2d(c, c, 3, padding=1)   # SSC or plain conv (ablation)
        self.attn = AdaptiveSparseLinearAttention(c, keep_ratio) if use_attn else None
        self.n2 = nn.GroupNorm(1, c) if use_attn else None
        self.n3 = nn.GroupNorm(1, c)
        self.ffn = nn.Sequential(nn.Conv2d(c, 2 * c, 1), nn.GELU(), nn.Conv2d(2 * c, c, 1))   # adds non-linearity
    def forward(self, x):
        x = x + self.spatial(self.n1(x))                              # local branch (residual)
        if self.attn is not None: x = x + self.attn(self.n2(x))       # global branch (residual)
        return x + self.ffn(self.n3(x))

class SRNet(nn.Module):
    def __init__(self, cfg, use_ssc=True, use_attn=True):
        super().__init__()
        c, self.scale = cfg.feat, cfg.scale
        self.head = nn.Conv2d(3, c, 3, padding=1)                     # RGB -> features
        self.body = nn.Sequential(*[Block(c, use_ssc, use_attn, cfg.keep_ratio) for _ in range(cfg.blocks)])
        self.tail = nn.Conv2d(c, c, 3, padding=1)
        up = []
        for _ in range(int(math.log2(cfg.scale))): up += [nn.Conv2d(c, 4 * c, 3, padding=1), nn.PixelShuffle(2)]   # 2x, twice
        self.up = nn.Sequential(*up, nn.Conv2d(c, 3, 3, padding=1))
    def forward(self, x):
        f = self.head(x)
        f = self.tail(self.body(f)) + f                               # long skip connection
        return self.up(f) + F.interpolate(x, scale_factor=self.scale, mode="bicubic", align_corners=False)  # network learns only the residual over bicubic

def n_params(m): return sum(p.numel() for p in m.parameters())

def sanity_check():                                                   # quick test: shapes, gradients and that the sparse mask really is sparse
    m = SRNet(cfg); x = torch.rand(2, 3, 28, 28)
    y = m(x); assert y.shape == (2, 3, 112, 112), y.shape
    y.mean().backward()
    a = m.body[0].attn
    assert a.score[0].weight.grad.abs().sum() > 0, "importance scorer gets no gradient"
    mask = a.key_mask(torch.randn(2, cfg.feat, 28, 28))
    assert abs(mask.round().mean().item() - cfg.keep_ratio) < 0.01, "mask is not top-k sparse"
    print("sanity OK | output", tuple(y.shape), "| params", f"{n_params(m):,}")
sanity_check()

In [ ]:
# CELL 6 - Metrics (PSNR/SSIM on the Y channel, as in the papers) and the Trainer class (L1 loss, AdamW, cosine LR, resume).
class Metrics:
    @staticmethod
    def rgb_to_y(x):                                                  # x: BCHW in [0,1] -> luminance Y in [0,1]
        return (16 + 65.481 * x[:, 0] + 128.553 * x[:, 1] + 24.966 * x[:, 2]) / 255.0
    @staticmethod
    def psnr_y(sr, hr, shave):                                        # per-image PSNR, border pixels cropped
        a, b = Metrics.rgb_to_y(sr)[:, shave:-shave, shave:-shave], Metrics.rgb_to_y(hr)[:, shave:-shave, shave:-shave]
        return 10 * torch.log10(1 / ((a - b) ** 2).flatten(1).mean(1).clamp_min(1e-10))
    @staticmethod
    def ssim_y(sr, hr, shave):
        from skimage.metrics import structural_similarity as ssim
        a, b = Metrics.rgb_to_y(sr)[:, shave:-shave, shave:-shave].numpy(), Metrics.rgb_to_y(hr)[:, shave:-shave, shave:-shave].numpy()
        return torch.tensor([ssim(p, q, data_range=1.0) for p, q in zip(a, b)])

@torch.no_grad()
def evaluate(fn, data, device, shave=4, bs=100, ssim=False):          # fn: any function LR-tensor -> SR-tensor (model or bicubic)
    if isinstance(fn, nn.Module): fn.eval()
    ps, ss = [], []
    for lr, hr in data.batches(bs):
        sr = fn(lr.to(device)).clamp(0, 1).cpu()                      # clamp: valid pixel range
        ps.append(Metrics.psnr_y(sr, hr, shave))
        if ssim: ss.append(Metrics.ssim_y(sr, hr, shave))
    out = {"PSNR_Y": torch.cat(ps).mean().item()}
    if ssim: out["SSIM_Y"] = torch.cat(ss).mean().item()
    return out

class Trainer:
    def __init__(self, model, cfg, loader, val_set, tag):
        self.model, self.cfg, self.loader, self.val_set, self.tag = model.to(device), cfg, loader, val_set, tag
        self.opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
        self.sched = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=cfg.epochs * len(loader), eta_min=1e-6)  # stepped every iteration
        d = Path(cfg.work_dir) / "ckpt"; d.mkdir(exist_ok=True)
        self.last, self.best = d / f"{tag}_last.pt", d / f"{tag}_best.pt"
        self.epoch, self.best_psnr, self.history = 0, -1.0, []
        if self.last.exists(): self._resume()                         # continue after a crash / laptop sleep
    def _save(self, path):
        torch.save({"model": self.model.state_dict(), "opt": self.opt.state_dict(), "sched": self.sched.state_dict(),
                    "epoch": self.epoch, "best": self.best_psnr, "hist": self.history}, path)
    def _resume(self):
        ck = torch.load(self.last, map_location=device)
        self.model.load_state_dict(ck["model"]); self.opt.load_state_dict(ck["opt"]); self.sched.load_state_dict(ck["sched"])
        self.epoch, self.best_psnr, self.history = ck["epoch"], ck["best"], ck["hist"]
        print(f"[{self.tag}] resumed from epoch {self.epoch}")
    def fit(self):
        for ep in range(self.epoch, self.cfg.epochs):
            self.model.train(); t0, total = time.time(), 0.0
            for lr, hr in self.loader:
                lr, hr = lr.to(device, non_blocking=True), hr.to(device, non_blocking=True)
                loss = F.l1_loss(self.model(lr), hr)                  # pixel loss against the clean HR face
                self.opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)   # keeps deformable offsets stable
                self.opt.step(); self.sched.step(); total += loss.item()
            val = evaluate(self.model, self.val_set, device)["PSNR_Y"]   # fixed val set -> comparable across epochs
            self.epoch = ep + 1; self.history.append({"epoch": self.epoch, "l1": total / len(self.loader), "val_psnr": val})
            print(f"[{self.tag}] epoch {self.epoch}/{self.cfg.epochs} | L1 {total / len(self.loader):.4f} | val PSNR-Y {val:.2f} dB | {time.time() - t0:.0f}s")
            if val > self.best_psnr: self.best_psnr = val; self._save(self.best)   # best-by-validation checkpoint
            self._save(self.last)
    def load_best(self):
        self.model.load_state_dict(torch.load(self.best, map_location=device)["model"])

In [ ]:
# CELL 7 - Run Part 1: train the baseline (plain conv) and our model with the SAME recipe, then test both ONCE on the fixed test set.
variants = {"baseline_conv": SRNet(cfg, use_ssc=False, use_attn=False),   # same width/depth, no SSC, no attention
            "ours_ssc_attn": SRNet(cfg)}                                   # sparse sampling conv + adaptive sparse linear attention
bicubic = lambda x: F.interpolate(x, scale_factor=cfg.scale, mode="bicubic", align_corners=False)
results = {"bicubic": {**evaluate(bicubic, test_set, device, ssim=True), "params": 0}}
for tag, model in variants.items():
    trainer = Trainer(model, cfg, train_loader, val_set, tag); trainer.fit(); trainer.load_best()
    results[tag] = {**evaluate(model, test_set, device, ssim=True), "params": n_params(model)}
print(f"{'model':16s}{'params':>10s}{'PSNR-Y':>9s}{'SSIM-Y':>9s}")
for k_, r in results.items(): print(f"{k_:16s}{r['params']:>10,}{r['PSNR_Y']:>9.2f}{r['SSIM_Y']:>9.4f}")
json.dump(results, open(Path(cfg.work_dir) / "part1_results.json", "w"), indent=2)

In [ ]:
# CELL 8 - (Part 2 preparation, run when ready) Download QMUL-SurvFace from the official Google Drive folders.
!pip install -q gdown
!gdown --folder https://drive.google.com/drive/folders/13ch6BPaexlKt8gXB_I8aX7p1G3yPm2Bl -O {cfg.work_dir}/qmul_main
!gdown --folder https://drive.google.com/drive/folders/10eLJIUpMTER_27tEjOvXDo-93FqJAvdB -O {cfg.work_dir}/qmul_test_images
# afterwards unzip the archives inside those folders into {cfg.work_dir}/QMUL-SurvFace (same layout as before)